In [1]:
import sys

class SudokuCSP:
    def __init__(self, filename):
        self.backtrack_calls = 0
        self.backtrack_failures = 0
        self.grid = self.load_board(filename)
        # Variables: (row, col) coordinates [cite: 76]
        # Domains: { (r,c): [1...9] } [cite: 183]
        self.domains = {}
        for r in range(9):
            for c in range(9):
                val = self.grid[r][c]
                if val == 0:
                    self.domains[(r, c)] = list(range(1, 10))
                else:
                    self.domains[(r, c)] = [val]

    def load_board(self, filename):
        """Reads board from text file based on assignment format[cite: 181, 182, 183]."""
        board = []
        with open(filename, 'r') as f:
            for line in f:
                if line.strip():
                    board.append([int(char) for char in line.strip()])
        return board

    def get_neighbors(self, var):
        """Finds all variables constrained by the current cell."""
        r, c = var
        neighbors = set()
        for i in range(9):
            if i != c: neighbors.add((r, i)) # Row
            if i != r: neighbors.add((i, c)) # Column
        # 3x3 Square
        start_r, start_c = 3 * (r // 3), 3 * (c // 3)
        for i in range(start_r, start_r + 3):
            for j in range(start_c, start_c + 3):
                if (i, j) != (r, c):
                    neighbors.add((i, j))
        return neighbors

    def ac3(self):
        """AC-3 algorithm for arc consistency."""
        queue = [(u, v) for u in self.domains for v in self.get_neighbors(u)]
        while queue:
            (u, v) = queue.pop(0)
            if self.revise(u, v):
                if len(self.domains[u]) == 0:
                    return False
                for w in self.get_neighbors(u):
                    if w != v:
                        queue.append((w, u))
        return True

    def revise(self, u, v):
        """Removes values from domain of u that don't satisfy constraint with v."""
        revised = False
        for x in self.domains[u][:]:
            # If v only has one value and it's x, u cannot be x
            if len(self.domains[v]) == 1 and x == self.domains[v][0]:
                self.domains[u].remove(x)
                revised = True
        return revised

    def is_consistent(self, var, val, assignment):
        """Checks if assignment satisfies binary constraints."""
        for neighbor in self.get_neighbors(var):
            if neighbor in assignment and assignment[neighbor] == val:
                return False
        return True

    def backtrack(self, assignment):
        """Backtracking search with forward checking[cite: 78, 196, 198]."""
        self.backtrack_calls += 1
        
        if len(assignment) == 81:
            return assignment

        # Select unassigned variable
        unassigned = [v for v in self.domains if v not in assignment]
        var = unassigned[0] 

        for value in self.domains[var]:
            if self.is_consistent(var, value, assignment):
                assignment[var] = value
                
                # Forward checking logic: basic consistency check
                result = self.backtrack(assignment)
                if result:
                    return result
                
                del assignment[var]

        self.backtrack_failures += 1
        return None

    def solve(self):
        """Initializes AC-3 and starts Backtracking[cite: 78]."""
        if not self.ac3():
            return None
        
        initial_assignment = {k: v[0] for k, v in self.domains.items() if len(v) == 1}
        return self.backtrack(initial_assignment)

    def print_solution(self, solution):
        """Prints the solved grid in 9x9 format[cite: 197]."""
        if not solution:
            print("No solution found.")
            return
        for r in range(9):
            row_str = "".join(str(solution[(r, c)]) for c in range(9))
            print(row_str)

# Main Execution
if __name__ == "__main__":
    files = ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]
    
    for file in files:
        try:
            print(f"--- Solving {file} ---")
            solver = SudokuCSP(file)
            result = solver.solve()
            solver.print_solution(result)
            print(f"Backtrack Calls: {solver.backtrack_calls}")
            print(f"Backtrack Failures: {solver.backtrack_failures}\n")
        except FileNotFoundError:
            print(f"Error: {file} not found. Please create the text file.")

--- Solving easy.txt ---
784932156
619485327
235176489
578261934
341897562
926543871
453729618
862314795
197658243
Backtrack Calls: 1
Backtrack Failures: 0

--- Solving medium.txt ---
875936142
169724385
243851679
452697831
986413257
731582964
517369428
628145793
394278516
Backtrack Calls: 95
Backtrack Failures: 69

--- Solving hard.txt ---
152346897
437189652
689572314
821637945
543891726
976425183
798253461
365914278
214768539
Backtrack Calls: 905
Backtrack Failures: 851

--- Solving veryhard.txt ---
431867925
652491387
897532164
384976512
519284736
276315849
943728651
765143298
128659473
Backtrack Calls: 10892
Backtrack Failures: 10836

